# 🛡️ Live Risk Control Panel
**Workflow**: Live Exposure Monitoring → Target Rebalancing → Dynamic Risk Limits → **Emergency Kill-Switch**

---

In [ ]:
from qtrader.output.analyst import AnalystSession, RoleContext
import polars as pl
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML
import plotly.graph_objects as go

session = AnalystSession(role=RoleContext.TRADER)
PLOTLY_DARK = dict(template="plotly_dark", plot_bgcolor='#0f1117', paper_bgcolor='#0f1117')

# Global State Simulation
state = {'net_pos': 2.4, 'max_pos': 5.0, 'var_limit': 0.02, 'engine_active': True}

## 1. Live Exposure & Limit Monitor
Visualizing current standing against enterprise risk constraints.

In [ ]:
def render_risk_gauge():
    fig = go.Figure(go.Indicator(
        mode = "gauge+number+delta",
        value = state['net_pos'],
        domain = {'x': [0, 1], 'y': [0, 1]},
        title = {'text': "Net Position (BTC)", 'font': {'size': 20}},
        delta = {'reference': 4.5, 'increasing': {'color': "#ef4444"}},
        gauge = {
            'axis': {'range': [None, 6.0], 'tickwidth': 1, 'tickcolor': "#94a3b8"},
            'bar': {'color': "#8b5cf6"},
            'steps': [
                {'range': [0, 3], 'color': "#064e3b"},
                {'range': [3, 4.5], 'color': "#78350f"},
                {'range': [4.5, 6], 'color': "#7f1d1d"}
            ],
            'threshold': {
                'line': {'color': "red", 'width': 4},
                'thickness': 0.75,
                'value': state['max_pos']
            }
        }
    ))
    fig.update_layout(height=400, **PLOTLY_DARK)
    fig.show()

render_risk_gauge()

## 2. Interactive Target Rebalancing
Manually adjust intended target exposure and review rebalance delta.

In [ ]:
target_slider = widgets.FloatSlider(value=state['net_pos'], min=0, max=5.0, step=0.1, description='Target BTC:')
rebalance_btn = widgets.Button(description="Review Rebalance", button_style='info')
output = widgets.Output()

def on_rebalance_click(b):
    with output:
        output.clear_output()
        delta = target_slider.value - state['net_pos']
        print(f"Current: {state['net_pos']} BTC")
        print(f"Target:  {target_slider.value} BTC")
        print(f"Required Action: {'BUY' if delta > 0 else 'SELL'} {abs(delta):.2f} BTC")
        print("--- Waiting for confirmation ---")

rebalance_btn.on_click(on_rebalance_click)
display(widgets.HBox([target_slider, rebalance_btn]), output)

## 3. 🚨 Emergency Controls (Kill-Switch)
Triggering this button will halt the live engine, cancel all working orders, and market-sell positions to flatten (Simulation).

In [ ]:
kill_btn = widgets.Button(
    description='EMERGENCY KILL-SWITCH',
    button_style='danger',
    layout=widgets.Layout(width='100%', height='80px')
)

def on_kill_signal(b):
    print("CRITICAL ALERT: KILL-SWITCH TRIGGERED")
    print("1. Halting QTrader Engine...")
    print("2. Canceling all active limit orders...")
    print("3. Flattening positions (Market SELL active)....")
    print("SYSTEM SHUT DOWN COMPLETE.")
    display(HTML("<h1 style='color:red;'>ENGINE HALTED</h1>"))

kill_btn.on_click(on_kill_signal)
display(kill_btn)